# Module 4 — Pattern Mining

1. Transactional encoding of the sellers.
2. Frequent itemsets across `min_sup` (frequent, closed and maximal).
3. Association rules across `min_conf` (counts, confidence/lift distributions, qualitative reading).
4. Rule exploitation:
   - (a) a rule-based classifier for `seller_rating_class`, trained only on the Module 3 training split and compared with the Module 3 models on the same test split;
   - (b) rule-based imputation of masked values;
   - (c) profiling the Module 2 segments with rules.

Algorithm: **Apriori** (`mlxtend` 0.23.4). FP-Growth gives identical itemsets, but its `mlxtend` implementation was ~70× slower on this dense data (229 s vs 3 s at `min_sup` = 0.05, measured in exploration), while the vectorized Apriori is fast with only ~50 distinct items.

In [1]:
import sys, json, time
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import plot_style as ps

ps.setup()
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
SEED = 42
CLASSES = ps.CLASS_ORDER

df = pd.read_csv(ROOT / "data" / "processed" / "clean_base.csv", index_col="sp_id")
split = pd.read_csv(ROOT / "data" / "processed" / "split.csv", index_col="sp_id")["set"].loc[df.index]
clusters = pd.read_csv(ROOT / "data" / "processed" / "cluster_labels.csv", index_col="sp_id")["kmeans4"].loc[df.index]
TARGET = "seller_rating_class"
TR, TE = df.index[split == "train"], df.index[split == "test"]
print(df.shape, "| train", len(TR), "| test", len(TE))

# Ordinal blue ramp (validated palette steps) for ordered series such as itemset length k or min_sup levels
RAMP = ["#b7d3f6", "#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#184f95", "#104281", "#0d366b"]

(13843, 31) | train 11074 | test 2769


## 1. Transactional encoding

Each seller becomes one transaction, with one item per attribute (`attribute=value`):

| Attribute | Items | Binning and justification |
|---|---|---|
| `n_listings` → `size`, `price`, `mileage`, `daysonmarket` → `dom`, `horsepower` → `hp`, `listing_span_days` → `span`, `pct_isCab` → `cab` | `low` / `mid` / `high` | **Equal-frequency tertiles.** The variables are heavily skewed (Module 1), so equal-width bins would put almost everyone in one bin. Tertiles give every level ≈ 33% support, so no item is frequent merely because of its bin width. |
| `savings_amount_median` → `savings` | `none` / `low` / `high` | **Domain bin.** 37% of sellers have zero savings, which is structural (new cars), so `none` is its own level and the positive values are split at their median. |
| `pct_salvage`, `pct_theft_title` | `salvage=yes`, `theft=yes` | **Presence items.** The distributions are zero-inflated (91% / 85% zeros), so only "has at least one such vehicle" carries information. |
| body type, listing colour, transmission, franchise status, new/used profile | e.g. `body=sedan`, `dealer=franchise`, `inventory=mixed` | Module 1 groupings of rare levels. `color=unknown` is kept, because its missingness is informative. |
| `seller_rating_class` | `rating=low/medium/high` | target item |

- **Missing values:** a missing value produces *no* item for that attribute. Nothing is imputed for mining.
- **Near-constant items:** items with support > 0.9 (`trans=a`, 94%) are dropped. They appear in almost every pattern, double the itemset count, and have lift ≈ 1 with everything.
- **Bin edges:** fit on the data the patterns are mined from. That is all sellers for the descriptive analysis (§2–3, unsupervised), and the training split or training fold only for the predictive uses (§4).

In [2]:
TERTILE = {"n_listings": "size", "price_median": "price", "mileage_median": "mileage", "daysonmarket_median": "dom",
           "horsepower_median": "hp", "listing_span_days": "span", "pct_isCab": "cab"}
CATEG = {"dominant_body_type": "body", "dominant_listing_color": "color", "dominant_transmission": "trans",
         "franchise_status": "dealer", "new_used_profile": "inventory"}

def fit_encoder(ref, max_support=0.9):
    """Learn bin edges and the item vocabulary from a reference set of sellers."""
    enc = {"edges": {c: ref[c].quantile([1 / 3, 2 / 3]).values for c in TERTILE},
           "sav_med": ref.savings_amount_median[ref.savings_amount_median > 0].median()}
    items = encode_items(ref, enc)
    sup = items.mean()
    enc["vocab"] = sup.index[sup <= max_support].tolist()
    enc["dropped"] = sup[sup > max_support].round(3).to_dict()
    return enc

def encode_items(d, enc, rating=True, extra=None):
    """Return a boolean seller x item matrix (one item per attribute, none if missing)."""
    cols = {}
    for c, n in TERTILE.items():
        lo, hi = enc["edges"][c]
        v = d[c]
        cols[n] = np.select([v <= lo, v <= hi, v > hi], [f"{n}=low", f"{n}=mid", f"{n}=high"], default=None)
    s = d.savings_amount_median
    cols["savings"] = np.select([s == 0, s <= enc["sav_med"], s > enc["sav_med"]], ["savings=none", "savings=low", "savings=high"], default=None)
    cols["salvage"] = np.where(d.pct_salvage > 0, "salvage=yes", None)
    cols["theft"] = np.where(d.pct_theft_title > 0, "theft=yes", None)
    for c, n in CATEG.items():
        cols[n] = np.where(d[c].notna(), n + "=" + d[c].astype(str), None)
    if rating:
        cols["rating"] = "rating=" + d[TARGET].astype(str)
    if extra is not None:
        for n, v in extra.items():
            cols[n] = v
    it = pd.DataFrame(cols, index=d.index)
    B = pd.get_dummies(it, prefix="", prefix_sep="").astype(bool)
    B = B.T.groupby(level=0).any().T  # identical names from different columns cannot occur, but keep it safe
    if "vocab" in enc:
        keep = enc["vocab"] + [c for c in B.columns if c.startswith("rating=") or (extra is not None and c.split("=")[0] in extra)]
        B = B.reindex(columns=sorted(set(keep)), fill_value=False)
    return B

enc_all = fit_encoder(df)
B = encode_items(df, enc_all)
print(B.shape[1], "items | dropped (support > 0.9):", enc_all["dropped"], "| mean items per seller: %.2f" % B.sum(1).mean())
item_sup = B.mean().sort_values(ascending=False)
item_sup.round(3).to_frame("support").T

50 items | dropped (support > 0.9): {'trans=a': 0.937} | mean items per seller: 13.30


,body=suv_crossover,rating=medium,inventory=used_only,dealer=franchise,dealer=independent,inventory=mixed,color=white,savings=none,hp=mid,dom=low,price=low,span=low,size=low,cab=low,cab=high,...,theft=yes,color=unknown,body=pickup_truck,salvage=yes,color=silver,color=gray,trans=cvt,color=red,color=blue,body=coupe_convertible,body=van_minivan,body=hatchback_wagon,color=other,inventory=new_only,trans=m
support,0.644,0.547,0.533,0.506,0.494,0.464,0.373,0.368,0.347,0.34,0.336,0.335,0.334,0.334,0.333,...,0.152,0.109,0.103,0.09,0.072,0.063,0.061,0.03,0.027,0.017,0.009,0.008,0.003,0.002,0.001


## 2. Frequent itemsets
### 2.1 Quantitative analysis: how many patterns per `min_sup` and length k

`min_sup` ∈ {0.30, 0.20, 0.15, 0.10, 0.075, 0.05, 0.03, 0.02}. Below 0.02 the number of itemsets explodes (> 40k at 0.02) and Apriori's candidate matrix exceeds memory without a length cap.

For each level we also count **closed** itemsets (no superset with the same support) and **maximal** itemsets (no frequent superset).

In [3]:
def closed_maximal(fi, n):
    cnt = dict(zip(fi.itemsets, np.rint(fi.support * n).astype(int)))
    has_super, has_equal = set(), set()
    for s, c in cnt.items():
        if len(s) > 1:
            for it in s:
                sub = s - {it}
                has_super.add(sub)
                if cnt[sub] == c:
                    has_equal.add(sub)
    closed = np.array([s not in has_equal for s in fi.itemsets])
    maximal = np.array([s not in has_super for s in fi.itemsets])
    return closed, maximal

MINSUPS = [0.30, 0.20, 0.15, 0.10, 0.075, 0.05, 0.03, 0.02]
fi_by, rows = {}, []
for ms in MINSUPS:
    t0 = time.time()
    fi = apriori(B, min_support=ms, use_colnames=True)
    fi["k"] = fi.itemsets.map(len)
    fi["closed"], fi["maximal"] = closed_maximal(fi, len(B))
    fi_by[ms] = fi
    rows.append({"min_sup": ms, "frequent": len(fi), "closed": int(fi.closed.sum()), "maximal": int(fi.maximal.sum()),
                 "max k": fi.k.max(), "seconds": round(time.time() - t0, 1), **{f"k={k}": int((fi.k == k).sum()) for k in range(1, 11)}})
fi_sweep = pd.DataFrame(rows).set_index("min_sup")
fi_sweep.to_csv(ps.TAB_DIR / "m4_itemset_sweep.csv")
fi_sweep

,frequent,closed,maximal,max k,seconds,k=1,k=2,k=3,k=4,k=5,k=6,k=7,k=8,k=9,k=10
min_sup,,,,,,,,,,,,,,,
0.300,59,59,27,4,0.0,32,18,8,1,0,0,0,0,0,0
0.200,179,177,42,5,0.0,34,73,47,21,4,0,0,0,0,0
0.150,431,421,106,6,0.1,36,167,131,73,22,2,0,0,0,0
0.100,1376,1296,291,7,0.4,38,379,457,331,137,31,3,0,0,0
0.075,2830,2610,474,8,0.7,39,477,932,821,425,118,17,1,0,0
0.050,6835,6061,1042,8,2.0,42,559,1946,2151,1425,577,125,10,0,0
0.030,19516,16181,2552,9,6.0,43,661,3687,6052,5282,2790,861,133,7,0
0.020,39961,31488,4817,10,15.2,44,735,4856,11282,11851,7509,2955,659,68,2


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.3))
ax = axes[0]
for i, (col, ls) in enumerate([("frequent", "-"), ("closed", "--"), ("maximal", ":")]):
    ax.plot(fi_sweep.index, fi_sweep[col], marker="o", markersize=3.5, color=ps.CATEGORICAL[i], linestyle=ls, label=col)
ax.set_yscale("log"); ax.set_xlabel("min_sup"); ax.set_ylabel("number of itemsets (log)")
ax.axvline(0.05, color=ps.NEUTRAL, linestyle=":", linewidth=1); ax.text(0.052, 30, "chosen\n0.05", fontsize=6, color=ps.INK_2)
ax.set_title("Itemsets vs min_sup"); ax.legend(frameon=False)
ax = axes[1]
show = [0.20, 0.10, 0.05, 0.03, 0.02]
ORD5 = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]  # ramp steps 250/350/450/550/700 (ordinal, light -> dark)
for j, ms in enumerate(show):
    cnt = fi_by[ms].k.value_counts().sort_index()
    ax.plot(cnt.index, cnt.values, marker="o", markersize=3, color=ORD5[j], label=f"min_sup={ms}")
ax.set_yscale("log"); ax.set_xlabel("itemset length k"); ax.set_ylabel("frequent itemsets (log)")
ax.set_title("Frequent itemsets by length k"); ax.legend(frameon=False, fontsize=6, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
ps.save(fig, "m4_itemsets_sweep")

**Reading the sweep.**
- **Growth:** the number of frequent itemsets grows roughly exponentially as `min_sup` decreases (log scale), and the length distribution shifts to longer patterns.
- **Why tertile items behave this way:** every tertile item has ≈ 33% support, so long conjunctions of "typical" levels stay frequent. At low support most new itemsets are long combinations of such items.
  - **Closed vs. frequent:** closed itemsets are 79–100% of the frequent ones (89% at 0.05). Few itemsets share exactly the support of a superset, because the tertile items do not imply each other.
  - **Maximal itemsets** are 5–8× fewer than frequent ones at `min_sup` ≤ 0.1 (1,042 vs 6,835 at 0.05). They summarize the *border* of the frequent region.
- **Chosen `min_sup` = 0.05**, i.e. ≥ 692 sellers per pattern:
  - *Statistical reliability:* every pattern describes hundreds of sellers.
  - *Coverage:* the minority class item `rating=low` (18.6%) can still combine with 2–3 other items.
  - *Manageability:* the 0.05 level yields a few thousand itemsets, whereas 0.03 and 0.02 triple and sextuple the count, mostly with redundant long combinations.

### 2.2 Qualitative analysis: frequent seller profiles

In [5]:
MS = 0.05
fi = fi_by[MS]
fmt = lambda s: " ∧ ".join(sorted(s))
top_long = fi[(fi.k >= 4) & fi.closed].sort_values("support", ascending=False).head(8)
top_long = top_long.assign(itemset=top_long.itemsets.map(fmt))[["itemset", "support", "k"]]
rating_sets = fi[fi.itemsets.map(lambda s: any(i.startswith("rating=") for i in s)) & (fi.k >= 3) & fi.closed]
rating_sets = rating_sets.assign(rating=rating_sets.itemsets.map(lambda s: [i for i in s if i.startswith("rating=")][0]))
top_rating = (rating_sets.sort_values("support", ascending=False).groupby("rating").head(3)
              .assign(itemset=lambda d: d.itemsets.map(fmt))[["rating", "itemset", "support", "k"]].sort_values(["rating", "support"], ascending=[True, False]))
maxi = fi[fi.maximal].sort_values(["k", "support"], ascending=False).head(5).assign(itemset=lambda d: d.itemsets.map(fmt))[["itemset", "support", "k"]]
top_long.to_csv(ps.TAB_DIR / "m4_top_itemsets.csv", index=False)
top_rating.to_csv(ps.TAB_DIR / "m4_top_rating_itemsets.csv", index=False)
print("Most frequent closed itemsets with k >= 4:"); print(top_long.to_string(index=False))
print("\nLongest maximal itemsets:"); print(maxi.to_string(index=False))
top_rating

Most frequent closed itemsets with k >= 4:
                                                                             itemset  support  k
                     dealer=franchise ∧ inventory=mixed ∧ mileage=low ∧ savings=none 0.313588  4
                 dealer=independent ∧ inventory=used_only ∧ mileage=high ∧ price=low 0.296829  4
              body=suv_crossover ∧ dealer=franchise ∧ inventory=mixed ∧ savings=none 0.271906  4
               body=suv_crossover ∧ dealer=franchise ∧ inventory=mixed ∧ mileage=low 0.262804  4
                  body=suv_crossover ∧ dealer=franchise ∧ mileage=low ∧ savings=none 0.251102  4
                   body=suv_crossover ∧ inventory=mixed ∧ mileage=low ∧ savings=none 0.249585  4
body=suv_crossover ∧ dealer=franchise ∧ inventory=mixed ∧ mileage=low ∧ savings=none 0.249512  5
                       dealer=franchise ∧ inventory=mixed ∧ savings=none ∧ size=high 0.248212  4

Longest maximal itemsets:
                                                         

,rating,itemset,support,k
1858,rating=high,dealer=independent ∧ inventory=used_only ∧ rating=high,0.160587,3
2387,rating=high,inventory=used_only ∧ rating=high ∧ size=low,0.129235,3
1908,rating=high,dealer=independent ∧ rating=high ∧ size=low,0.125551,3
1725,rating=low,dealer=franchise ∧ inventory=mixed ∧ rating=low,0.095933,3
813,rating=low,body=suv_crossover ∧ dealer=franchise ∧ rating=low,0.082135,3
1859,rating=low,dealer=independent ∧ inventory=used_only ∧ rating=low,0.079607,3
1726,rating=medium,dealer=franchise ∧ inventory=mixed ∧ rating=medium,0.269234,3
1860,rating=medium,dealer=independent ∧ inventory=used_only ∧ rating=medium,0.249151,3
814,rating=medium,body=suv_crossover ∧ dealer=franchise ∧ rating=medium,0.242072,3


**Frequent seller profiles.** The most frequent long itemsets rediscover the two dominant seller types of Module 2, without any distance computation:
- `dealer=franchise ∧ inventory=mixed ∧ mileage=low ∧ savings=none` (31% of sellers) is the **franchise new-car dealer** (K-Means C0). The longest maximal itemsets extend it with SUV, high price, large size and `rating=medium` (8 items, 7.7%).
- `dealer=independent ∧ inventory=used_only ∧ mileage=high ∧ price=low` (30%) is the **budget used-car dealer** (C3).

Itemsets containing a rating item mostly reflect class frequencies (`rating=medium` appears with both seller types). Their *composition* still differs:
- `rating=high` co-occurs most with `dealer=independent ∧ inventory=used_only` (16%) and with `size=low` (13%).
- `rating=low` co-occurs most with `dealer=franchise ∧ inventory=mixed` (9.6%).

Whether these co-occurrences are stronger than chance is measured by the lift of the rules in §3.

## 3. Association rules
### 3.1 Quantitative analysis

Rules are generated from the `min_sup` = 0.05 itemsets with confidence ≥ 0.3 (the floor of the sweep), then counted for `min_conf` ∈ {0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95} by rule length k = |antecedent| + |consequent|.

In [6]:
t0 = time.time()
rules = association_rules(fi[["support", "itemsets"]], num_itemsets=len(B), metric="confidence", min_threshold=0.3,
                          return_metrics=["antecedent support", "consequent support", "support", "confidence", "lift"])
rules["k"] = rules.antecedents.map(len) + rules.consequents.map(len)
print(f"{len(rules):,} rules with confidence >= 0.3 ({time.time() - t0:.1f}s)")
MINCONFS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
rule_sweep = pd.DataFrame({mc: rules[rules.confidence >= mc].k.value_counts().sort_index() for mc in MINCONFS}).T.fillna(0).astype(int)
rule_sweep["total"] = rule_sweep.sum(axis=1)
rule_sweep["lift > 1 %"] = [round((rules[rules.confidence >= mc].lift > 1).mean() * 100, 1) for mc in MINCONFS]
rule_sweep.index.name = "min_conf"
rule_sweep.to_csv(ps.TAB_DIR / "m4_rule_sweep.csv")
rule_sweep

87,266 rules with confidence >= 0.3 (0.6s)


k,2,3,4,5,6,7,8,total,lift > 1 %
min_conf,,,,,,,,,
0.30,818,6367,17739,26823,23314,10487,1718,87266,97.2
0.40,346,4187,12923,20149,17620,7833,1224,64282,98.2
0.50,212,2876,9306,14831,12920,5672,838,46655,98.7
0.60,109,1817,6882,11175,9603,4068,579,34233,99.8
0.70,57,1143,5057,8193,6827,2719,321,24317,100.0
0.80,34,712,3369,5232,4058,1407,157,14969,100.0
0.90,21,444,2300,3675,2881,1040,130,10491,100.0
0.95,10,287,1670,2584,1772,578,70,6971,100.0


In [7]:
MC = 0.6
R = rules[rules.confidence >= MC].copy()
fig, axes = plt.subplots(1, 4, figsize=(ps.FULL_W, 2.2), gridspec_kw={"width_ratios": [1.7, 1, 1, 1.15]})
ax = axes[0]
# heatmap (k x min_conf) with the counts written in: 7 overlapping lines would not be distinguishable
from matplotlib.colors import LinearSegmentedColormap, LogNorm
seq = LinearSegmentedColormap.from_list("seq_blue", ["#ffffff", "#cde2fb", "#6da7ec", "#2a78d6", "#104281"])
ks = [k for k in rule_sweep.columns if isinstance(k, (int, np.integer))]
H = rule_sweep[ks].T.to_numpy()
ax.imshow(np.maximum(H, 1), cmap=seq, norm=LogNorm(vmin=1, vmax=H.max()), aspect="auto")
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H[i, j]
        lab = f"{v / 1000:.0f}k" if v >= 10000 else (f"{v / 1000:.1f}k" if v >= 1000 else str(v))
        ax.text(j, i, lab, ha="center", va="center", fontsize=5,
                color="white" if v > 3000 else ps.INK)
ax.set_xticks(range(len(MINCONFS))); ax.set_xticklabels([f"{m:.2f}".rstrip("0").rstrip(".") for m in MINCONFS], fontsize=5.5)
ax.set_yticks(range(len(ks))); ax.set_yticklabels(ks, fontsize=6)
ax.set_xlabel("min_conf"); ax.set_ylabel("rule length k"); ax.set_title("Rules by min_conf and k", fontsize=7.5)
ax.grid(False)
for s_ in ax.spines.values(): s_.set_visible(False)
ax = axes[1]
ax.hist(R.confidence, bins=30, color=ps.CATEGORICAL[0])
ax.set_xlabel("confidence"); ax.set_ylabel("rules"); ax.set_title(f"Confidence (min_conf={MC})", fontsize=7.5)
ax = axes[2]
ax.hist(R.lift, bins=40, color=ps.CATEGORICAL[1])
ax.axvline(1, color=ps.INK_2, linewidth=0.8, linestyle="--")
ax.set_xlabel("lift"); ax.set_title("Lift", fontsize=7.5)
ax = axes[3]
smp = R.sample(min(len(R), 6000), random_state=SEED)
sc = ax.scatter(smp.support, smp.confidence, c=smp.lift, s=2, cmap=seq, vmin=1, vmax=4, rasterized=True)
cb = fig.colorbar(sc, ax=ax, pad=0.02); cb.set_label("lift", fontsize=6.5); cb.outline.set_visible(False)
ax.set_xlabel("support"); ax.set_ylabel("confidence"); ax.set_title("Support–confidence (sample)", fontsize=7.5)
fig.tight_layout()
ps.save(fig, "m4_rules_quant")
print(R[["confidence", "lift"]].describe().round(3).T)
print("share of rules with lift > 1.5:", round((R.lift > 1.5).mean(), 3), "| with lift < 1:", round((R.lift < 1).mean(), 3))

              count   mean    std    min    25%    50%    75%    max
confidence  34233.0  0.799  0.130  0.600  0.686  0.776  0.926  1.000
lift        34233.0  2.305  0.583  0.937  1.978  2.228  2.660  6.411
share of rules with lift > 1.5: 0.924 | with lift < 1: 0.002


**Reading the rule statistics.**
- **Rule counts:** at `min_sup` = 0.05 there are 87,266 rules with confidence ≥ 0.3. Most have length k = 5–6, because long frequent itemsets generate many antecedent/consequent splits (an itemset of size k gives 2^k − 2 rules).
- **Effect of `min_conf`:** the count falls slowly with `min_conf`, only 12.5× from 0.3 to 0.95. Almost 7,000 rules have confidence ≥ 0.95, because the encoding contains near-deterministic relations: a mixed franchise inventory almost always means `savings=none` and low mileage, and old low-power stock almost always means `price=low`.
- **Chosen `min_conf` = 0.6:** 34,233 rules remain. Above 0.6, *every* rule has lift > 1 (99.8% at 0.6), so the threshold removes the rules whose confidence merely reflects a frequent consequent.
- **Confidence distribution** (≥ 0.6): spread over [0.6, 1] with a median of 0.78 and a secondary peak at 1.0 (the deterministic relations).
- **Lift distribution:** concentrated between 2 and 3 (median 2.2; 92% of rules above 1.5). Most single-item consequents are tertile items with support ≈ 1/3, so their lift is capped near 3. The tail up to 6.4 comes from **multi-item consequents** (e.g. `mileage=low ∧ price=high`), whose lower joint support raises the lift ceiling.
- **Support–confidence scatter:** high-confidence rules exist at all support levels, while the highest lifts sit at low support. This is the usual trade-off between generality and specificity.

**Note on algorithm cost.** The whole sweep (8 support levels, up to ~40k itemsets) takes under a minute with Apriori, which here is faster than the `mlxtend` FP-Growth implementation. This is because the item vocabulary is small (50 items) and the transactions are dense (13.3 items per seller). In that regime the vectorized candidate counting dominates FP-Growth's per-node Python overhead.

### 3.2 Qualitative analysis: the most interesting rules

Many rules with high confidence are trivial: their consequent is a frequent item, or they restate the definition of a variable (e.g. `inventory=used_only → savings≠none`). We therefore rank by **lift** and keep single-item consequents.

Two filters make the list readable:
- **Redundancy:** a rule is dropped if a rule with the same consequent and a *shorter* antecedent (a subset) reaches at least 99% of its lift. The longer rule adds conditions without adding information.
- **Diversity:** only the best rule *per consequent item* is shown. Otherwise the top of the list is dominated by variants of the same relation (e.g. dozens of `… → price=low` rules).

In [8]:
def prune_redundant(r, tol=0.99):
    r = r.sort_values(["lift", "support"], ascending=False)
    by_cons = {}
    kept = []
    for _, row in r.iterrows():
        prev = by_cons.setdefault(row.consequents, [])
        if any(a < row.antecedents and l >= tol * row.lift for a, l in prev):
            continue
        prev.append((row.antecedents, row.lift))
        kept.append(row)
    return pd.DataFrame(kept)

def fmt_rule(r):
    return f"{fmt(r.antecedents)}  →  {fmt(r.consequents)}"

single = R[R.consequents.map(len) == 1]
cand = prune_redundant(single[single.lift >= 1.5])
top_lift = (cand.assign(cons=cand.consequents.map(fmt)).sort_values("lift", ascending=False)
            .groupby("cons").head(1).head(12))
top_lift = top_lift.assign(rule=top_lift.apply(fmt_rule, axis=1))[["rule", "support", "confidence", "lift"]]
top_lift.to_csv(ps.TAB_DIR / "m4_top_rules.csv", index=False)
top_lift.round(3)

,rule,support,confidence,lift
826,body=pickup_truck ∧ price=high → hp=high,0.052,0.960,3.005
25613,body=sedan ∧ hp=low ∧ mileage=high ∧ size=low → price=low,0.053,0.999,2.976
61331,body=suv_crossover ∧ dealer=independent ∧ price=low ∧ savings=low ∧ size=low → mileage=high,0.052,0.961,2.885
49641,hp=high ∧ mileage=low ∧ savings=none ∧ span=high → price=high,0.061,0.961,2.883
4445,dealer=franchise ∧ savings=high → mileage=mid,0.051,0.949,2.847
67158,dealer=franchise ∧ dom=high ∧ inventory=mixed ∧ price=high ∧ savings=none → mileage=low,0.081,0.948,2.844
575,trans=cvt → hp=low,0.057,0.926,2.778
25017,dealer=independent ∧ hp=low ∧ inventory=used_only ∧ price=low → body=sedan,0.087,0.610,2.775
51483,mileage=high ∧ price=low ∧ rating=high ∧ span=low → size=low,0.050,0.902,2.701
21564,dom=high ∧ mileage=low ∧ size=high → savings=none,0.071,0.983,2.668


In [9]:
# Rules predicting the rating need a lower support (rating=low covers 18.6% of sellers): min_sup 0.01, antecedent <= 3 items
fi_r = apriori(B, min_support=0.01, use_colnames=True, max_len=4, low_memory=True)
rr = association_rules(fi_r, num_itemsets=len(B), metric="lift", min_threshold=1.0,
                       return_metrics=["support", "confidence", "lift"])
rr = rr[rr.consequents.map(lambda s: len(s) == 1 and next(iter(s)).startswith("rating="))]
rating_rules = []
for c in CLASSES:
    x = prune_redundant(rr[rr.consequents == frozenset([f"rating={c}"])].nlargest(300, "lift")).head(4)
    rating_rules.append(x.assign(rule=x.apply(fmt_rule, axis=1), base_rate=(df[TARGET] == c).mean()))
rating_rules = pd.concat(rating_rules)[["rule", "support", "confidence", "lift", "base_rate"]]
rating_rules.to_csv(ps.TAB_DIR / "m4_rating_rules.csv", index=False)
print({c: int((rr.consequents == frozenset([f'rating={c}'])).sum()) for c in CLASSES}, "rules with lift >= 1 per class")
rating_rules.round(3)

{'low': 1073, 'medium': 2299, 'high': 1529} rules with lift >= 1 per class


,rule,support,confidence,lift,base_rate
50407,body=sedan ∧ price=mid ∧ size=high → rating=low,0.011,0.399,2.141,0.186
260549,price=mid ∧ size=high ∧ trans=cvt → rating=low,0.012,0.398,2.137,0.186
256606,mileage=low ∧ price=mid ∧ span=high → rating=low,0.014,0.384,2.060,0.186
242906,hp=low ∧ size=high ∧ span=high → rating=low,0.016,0.379,2.033,0.186
259266,price=high ∧ savings=low ∧ size=high → rating=medium,0.015,0.770,1.407,0.547
238313,hp=high ∧ savings=low ∧ size=high → rating=medium,0.012,0.734,1.341,0.547
257530,mileage=mid ∧ price=high ∧ size=high → rating=medium,0.024,0.730,1.333,0.547
161516,color=black ∧ price=high ∧ savings=low → rating=medium,0.011,0.723,1.320,0.547
107448,cab=high ∧ price=mid ∧ size=low → rating=high,0.012,0.544,2.042,0.266
246172,hp=mid ∧ mileage=mid ∧ size=low → rating=high,0.014,0.529,1.986,0.266


**Reading the rules.**

*Top-lift rules (one per consequent)* describe the structure of inventories and are consistent with domain knowledge and Module 3:
- **Price determinants:** `body=pickup_truck ∧ price=high → hp=high` (conf 0.96, lift 3.0), and `sedan ∧ hp=low ∧ mileage=high ∧ size=low → price=low` (conf 0.999). Old, low-power stock is cheap, the same drivers found by the regression.
- **Inventory structure:** `trans=cvt → hp=low` (CVTs are fitted to small engines); franchise dealers that discount (`savings=high`) sell mid-mileage, late-model used cars.
- **Dealer types:** franchise mixed inventories without savings are low-mileage; independent budget sellers have high mileage.

*Rating rules* (min_sup 0.01, |A| ≤ 3) are much weaker:
- **`rating=high`:** confidence ≈ 0.53 (lift ≈ 2.0, base rate 0.27) for **small sellers** (`size=low`) with mid-price or mid-mileage stock. This is the "small seller" signal of the Decision Tree and of Module 2.
- **`rating=low`:** confidence ≈ 0.39 (lift ≈ 2.1, base rate 0.19) for **large sellers** (`size=high`) with mid-priced sedans, CVT or low-power stock, or very long listing spans.
- **`rating=medium`:** confidence 0.72–0.77 (lift ≤ 1.4) for large, expensive sellers with little discounting.
- **Weakness:** every rating rule has support of only 1–2%, and no rule reaches confidence above 0.55 for `low` or `high`. The rating is associated with seller size and price segment, but only weakly, which anticipates the classifier results in §4.1.

## 4. Exploiting the rules
### 4.1 Rule-based classifier for `seller_rating_class`

**Leak-free design.** The model uses the Module 3 split (`split.csv`): bin edges, the item vocabulary and the rules are learnt **on the training split only**. Hyperparameters are tuned by the same stratified 5-fold CV as in Module 3, re-encoding and re-mining inside each fold. The test split is used once.

**Class-association rules.** Every frequent antecedent A (non-rating items only, |A| ≤ L, support ≥ s on the training data) defines the three rules `A → rating=c`, one per class. Their confidences P(c | A) form the class distribution of the sellers matching A. Rules are "useful" only if lift = P(c | A) / P(c) > 1.

**How should matching rules be combined into one prediction?** This is the key design choice, so it is a hyperparameter with 8 variants:

| Aggregation | Prediction for a seller |
|---|---|
| `class-best-conf` (CBA-style) | per class, the highest confidence among matching rules with lift > 1; argmax |
| `class-best-lift` | the same with lift instead of confidence; argmax |
| `best-antecedent` | the matching antecedent with the highest max-lift (the *most discriminative* rule); predict its most confident class |
| `top10` | average class distribution of the 10 most discriminative matching antecedents; argmax |
| `mean` | average class distribution over *all* matching antecedents; argmax |
| `… + prior` (for the last three) | the same, but argmax of P(c)-normalized scores (class-prior correction, as for KNN in Module 3) |

The two `class-best-*` variants compare the classes by their *best* rule. These scores are not comparable across classes:
- Confidence is bounded by the class frequency, so `medium` (55%) wins most comparisons.
- Lift is bounded by 1/P(c): 1.8 for `medium` but 5.4 for `low`, and taking the maximum over thousands of rules favours classes with many noisy extreme rules.

The antecedent-based variants avoid this, because all three confidences come from the *same* group of sellers.

**Uncovered sellers** (no matching rule or antecedent) get the training majority class. Coverage is reported.

**Grid:** s ∈ {0.005, 0.01, 0.02} × L ∈ {1, 2, 3, 4} × 8 aggregations, which gives 96 configurations, each scored by 5-fold CV. Mining is done once per (fold, s, L); the aggregations are evaluated on the same rules. In exploration, two more settings were checked: L = 5 (s = 0.01) gained only +0.003 over L = 4, and s = 0.0025 was worse than s = 0.005. So the optimum is not cut off by the grid; L = 5 at s = 0.005 was too slow to be practical.

In [10]:
AGGS = ["class-best-conf", "class-best-lift", "best-antecedent", "best-antecedent + prior",
        "top10", "top10 + prior", "mean", "mean + prior"]

class RuleClassifier:
    """Class-association-rule classifier. fit() mines antecedents on the training sellers only."""

    def __init__(self, min_sup=0.005, max_ante=4, agg="best-antecedent"):
        self.min_sup, self.max_ante, self.agg = min_sup, max_ante, agg

    def fit(self, d):
        self.enc_ = fit_encoder(d)
        Bt = encode_items(d, self.enc_, rating=False)
        fi = apriori(Bt, min_support=self.min_sup, use_colnames=True, max_len=self.max_ante, low_memory=True)
        self.antecedents_ = list(fi.itemsets)
        self.items_ = list(Bt.columns)
        col = {c: i for i, c in enumerate(self.items_)}
        A = np.zeros((len(fi), len(self.items_)), dtype=np.int16)
        for j, s in enumerate(fi.itemsets):
            A[j, [col[i] for i in s]] = 1
        self.A_, self.alen_ = A, A.sum(1)
        match = (Bt.to_numpy(np.int16) @ A.T) == self.alen_                      # train sellers x antecedents
        Y = np.stack([(d[TARGET] == c).to_numpy() for c in CLASSES], 1).astype(float)
        cnt = match.T.astype(float) @ Y                                           # antecedent x class counts
        self.conf_ = cnt / cnt.sum(1, keepdims=True)                              # P(c | A) for the 3 rules A -> c
        self.prior_ = Y.mean(0)
        self.lift_ = self.conf_ / self.prior_
        self.n_rules_ = int((self.lift_ > 1).sum())                               # rules A -> c with lift > 1
        self.majority_ = CLASSES[int(self.prior_.argmax())]
        return self

    def _match(self, d):
        X = encode_items(d, self.enc_, rating=False).reindex(columns=self.items_, fill_value=False).to_numpy(np.int16)
        return (X @ self.A_.T) == self.alen_

    def predict(self, d, agg=None, match=None):
        agg = agg or self.agg
        M = self._match(d) if match is None else match
        if agg.startswith("class-best"):
            metric = self.conf_ if agg == "class-best-conf" else self.lift_
            metric = np.where(self.lift_ > 1, metric, -np.inf)                    # only rules with lift > 1
            score = np.stack([np.where(M, metric[:, k], -np.inf).max(1) for k in range(len(CLASSES))], 1)
            covered = np.isfinite(score).any(1)
        else:
            covered = M.any(1)
            disc = self.lift_.max(1)                                              # discriminativeness of each antecedent
            if agg.startswith("best-antecedent"):
                score = self.conf_[np.where(M, disc, -1).argmax(1)]
            elif agg.startswith("top10"):
                idx = np.argsort(-np.where(M, disc, -1), axis=1)[:, :10]
                w = np.take_along_axis(M, idx, 1)[..., None]
                score = (self.conf_[idx] * w).sum(1) / np.maximum(w.sum(1), 1)
            else:
                score = (M.astype(float) @ self.conf_) / np.maximum(M.sum(1, keepdims=True), 1)
            if agg.endswith("+ prior"):
                score = score / self.prior_
        self.covered_ = covered
        pred = np.array(CLASSES)[np.argmax(score, axis=1)]
        pred[~covered] = self.majority_
        return pred

dtr = df.loc[TR]
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)  # same folds as Module 3
folds = list(cv.split(dtr, dtr[TARGET]))
cv_rows = []
t0 = time.time()
for s, L in product([0.005, 0.01, 0.02], [1, 2, 3, 4]):
    per = {a: {"f1": [], "acc": [], "cov": []} for a in AGGS}
    nrules = []
    for a_idx, b_idx in folds:
        m = RuleClassifier(s, L).fit(dtr.iloc[a_idx])
        M = m._match(dtr.iloc[b_idx])
        yb = dtr[TARGET].iloc[b_idx].to_numpy()
        nrules.append(m.n_rules_)
        for agg in AGGS:
            p = m.predict(None, agg=agg, match=M)
            per[agg]["f1"].append(f1_score(yb, p, average="macro")); per[agg]["acc"].append(accuracy_score(yb, p))
            per[agg]["cov"].append(m.covered_.mean())
    for agg in AGGS:
        cv_rows.append({"min_sup": s, "max antecedent": L, "aggregation": agg, "CV macro-F1": np.mean(per[agg]["f1"]),
                        "CV macro-F1 std": np.std(per[agg]["f1"]), "CV accuracy": np.mean(per[agg]["acc"]),
                        "coverage": np.mean(per[agg]["cov"]), "rules lift>1 (mean)": np.mean(nrules)})
    print(f"s={s} L={L} done ({time.time() - t0:.0f}s)", flush=True)
rb_cv = pd.DataFrame(cv_rows).sort_values("CV macro-F1", ascending=False)
rb_cv.to_csv(ps.TAB_DIR / "m4_rule_classifier_cv.csv", index=False)
print(rb_cv.groupby("aggregation")["CV macro-F1"].max().sort_values(ascending=False).round(3))
rb_cv.head(10).round(3)

s=0.005 L=1 done (1s)


s=0.005 L=2 done (5s)


s=0.005 L=3 done (37s)


s=0.005 L=4 done (180s)


s=0.01 L=1 done (181s)


s=0.01 L=2 done (185s)


... (4 outputs left out)

s=0.02 L=3 done (333s)


s=0.02 L=4 done (388s)


aggregation
best-antecedent            0.435
top10                      0.420
mean + prior               0.400
top10 + prior              0.330
best-antecedent + prior    0.283
class-best-lift            0.283
class-best-conf            0.243
mean                       0.236
Name: CV macro-F1, dtype: float64


,min_sup,max antecedent,aggregation,CV macro-F1,CV macro-F1 std,CV accuracy,coverage,rules lift>1 (mean)
26,0.005,4,best-antecedent,0.435,0.007,0.486,1.0,46459.8
58,0.010,4,best-antecedent,0.426,0.012,0.508,1.0,31729.2
28,0.005,4,top10,0.420,0.007,0.525,1.0,46459.8
18,0.005,3,best-antecedent,0.412,0.010,0.523,1.0,10169.4
63,0.010,4,mean + prior,0.400,0.009,0.408,1.0,31729.2
31,0.005,4,mean + prior,0.399,0.008,0.405,1.0,46459.8
50,0.010,3,best-antecedent,0.398,0.010,0.532,1.0,8264.2
95,0.020,4,mean + prior,0.393,0.010,0.403,1.0,17432.8
60,0.010,4,top10,0.393,0.018,0.535,1.0,31729.2
55,0.010,3,mean + prior,0.389,0.011,0.394,1.0,8264.2


In [11]:
best = rb_cv.iloc[0]
rb = RuleClassifier(float(best.min_sup), int(best["max antecedent"]), best.aggregation).fit(dtr)
y_te = df.loc[TE, TARGET].to_numpy()
p_te = rb.predict(df.loc[TE])
rb_row = {"model": "Rule-based (M4)", "accuracy": accuracy_score(y_te, p_te), "macro-F1": f1_score(y_te, p_te, average="macro"),
          **{f"R {c}": r for c, r in zip(CLASSES, recall_score(y_te, p_te, labels=CLASSES, average=None))},
          "coverage": rb.covered_.mean()}
print("chosen:", dict(best[["min_sup", "max antecedent", "aggregation"]]), "| antecedents:", len(rb.antecedents_), "| rules with lift > 1:", rb.n_rules_)
m3 = pd.read_csv(ps.TAB_DIR / "m3_test_metrics.csv", index_col="model")
cmp = pd.concat([m3[["accuracy", "macro-F1", "R low", "R medium", "R high"]].assign(coverage=1.0),
                 pd.DataFrame([rb_row]).set_index("model")])
cmp.to_csv(ps.TAB_DIR / "m4_classifier_comparison.csv")
print(pd.DataFrame(confusion_matrix(y_te, p_te, labels=CLASSES), index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES]))
cmp.round(3)

chosen: {'min_sup': 0.005, 'max antecedent': 4, 'aggregation': 'best-antecedent'} | antecedents: 32362 | rules with lift > 1: 46323
             pred low  pred medium  pred high
true low          127          273        116
true medium       225          879        412
true high          86          296        355


,accuracy,macro-F1,R low,R medium,R high,coverage
model,,,,,,
Decision Tree,0.459,0.437,0.368,0.430,0.583,1.0
KNN,0.449,0.433,0.457,0.418,0.506,1.0
GaussianNB,0.440,0.407,0.374,0.480,0.402,1.0
CategoricalNB,0.459,0.414,0.238,0.493,0.544,1.0
Baseline: majority,0.547,0.236,0.000,1.000,0.000,1.0
Rule-based (M4),0.492,0.433,0.246,0.580,0.482,1.0


In [12]:
# Figure: test macro-F1 of all classifiers (M3 + M4) and the rule-based model's CV grid
fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.1), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
order = cmp["macro-F1"].sort_values().index
cols = [ps.CATEGORICAL[1] if n == "Rule-based (M4)" else (ps.NEUTRAL if n.startswith("Baseline") else ps.CATEGORICAL[0]) for n in order]
ax.barh(order, cmp.loc[order, "macro-F1"], color=cols, height=0.6)
for y_, v in enumerate(cmp.loc[order, "macro-F1"]):
    ax.text(v + 0.005, y_, f"{v:.3f}", va="center", fontsize=6.5, color=ps.INK_2)
ax.set_xlim(0, 0.52); ax.set_xlabel("test macro-F1"); ax.set_title("Rule-based vs Module 3 classifiers", fontsize=7.5)
ax.grid(axis="y", visible=False); ax.tick_params(axis="y", labelsize=6.5)
ax = axes[1]
SHOW_AGG = ["best-antecedent", "top10", "mean + prior", "class-best-conf", "class-best-lift"]
for i, agg in enumerate(SHOW_AGG):
    d_ = rb_cv[(rb_cv.aggregation == agg) & (rb_cv.min_sup == best.min_sup)].sort_values("max antecedent")
    ax.plot(d_["max antecedent"], d_["CV macro-F1"], marker="o", markersize=3, color=ps.CATEGORICAL[i],
            linestyle="-" if i < 3 else "--", label=agg)
ax.axhline(0.236, color=ps.NEUTRAL, linestyle=":", linewidth=1)
ax.text(4.0, 0.226, "majority baseline", fontsize=5.5, color=ps.INK_2, ha="right", va="center")
ax.set_ylim(0.215, None)
ax.set_xticks([1, 2, 3, 4]); ax.set_xlabel("max antecedent length L"); ax.set_ylabel("CV macro-F1")
ax.set_title(f"Rule classifier: CV by aggregation (s={best.min_sup})", fontsize=7.5, pad=22)
ax.legend(frameon=False, fontsize=5.2, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
fig.tight_layout()
ps.save(fig, "m4_rule_classifier")

**Rule-based classifier results.**
- **The aggregation choice matters more than any other parameter.** The CBA-style "best rule per class" variants barely beat the majority baseline (CV macro-F1 0.243 with confidence, 0.283 with lift). As argued above, per-class maxima are not comparable across classes, so they almost always pick `medium` (confidence) or `low`/`high` (lift).
- **The best variant uses the most discriminative matching antecedent** and predicts its majority class (`best-antecedent`): s = 0.005, L = 4, CV macro-F1 0.435 ± 0.007. Longer antecedents help steadily (L = 1 → 4) because they isolate more homogeneous groups of sellers; lower support helps because more specific antecedents can be found.
- **Prior correction hurts the "best antecedent" variant.** The most discriminative antecedent already has an extreme class distribution, and dividing by the prior over-promotes `low`.
- **Test set:** macro-F1 0.433 and accuracy 0.492, which ties the Decision Tree (0.437) and KNN (0.433) and gives the highest accuracy among the non-baseline models.
  - Its errors resemble CategoricalNB, which also works on binned features: strong on `medium` (recall 0.58), moderate on `high` (0.48) and weak on `low` (0.25).
  - Coverage is 100%, because every seller matches at least one single-item antecedent, so the default class is never needed.
- **Interpretability:** the model holds ≈ 46k rules with lift > 1, so it is not readable as a whole. Each *individual* prediction, however, is explained by one rule ("sellers with these 4 properties are 53% `high`").
- **Conclusion:** association rules reach the same ceiling as the Module 3 models. This confirms that the limit lies in the information the features carry about the rating, not in the learning algorithm.

### 4.2 Rule-based imputation of masked values

**Protocol.** Two attributes are imputed: the **price tertile** (a binned numeric attribute) and the **body type** (categorical).
- **Masking:** all test-split sellers have the attribute masked, so 2,769 masked values per attribute. That is a 20% mask, larger than the suggested 10%, which makes the estimate more stable.
- **Training:** rules `A → attribute=value` are mined on the training split (min_sup 0.01, |A| ≤ 3, confidence-ranked). The antecedent may use any other item except the rating.
- **Prediction:** each masked value is predicted by the highest-confidence matching rule. Uncovered sellers get the training mode.
- **Baseline:** the mode of the training split.

In [13]:
def impute_eval(attr_prefix, min_sup=0.01, max_ante=3):
    enc = fit_encoder(dtr)
    Btr = encode_items(dtr, enc, rating=False)
    Bte = encode_items(df.loc[TE], enc, rating=False).reindex(columns=Btr.columns, fill_value=False)
    targets = [c for c in Btr.columns if c.startswith(attr_prefix + "=")]
    truth = Bte[targets].idxmax(axis=1).where(Bte[targets].any(axis=1))
    fi = apriori(Btr, min_support=min_sup, use_colnames=True, max_len=max_ante + 1, low_memory=True)
    r = association_rules(fi, num_itemsets=len(Btr), metric="confidence", min_threshold=0.0, return_metrics=["support", "confidence", "lift"])
    r = r[r.consequents.map(lambda s: len(s) == 1 and next(iter(s)) in targets)
          & r.antecedents.map(lambda s: not any(i.startswith(attr_prefix + "=") for i in s))].reset_index(drop=True)
    others = [c for c in Btr.columns if c not in targets]
    col = {c: i for i, c in enumerate(others)}
    A = np.zeros((len(r), len(others)), dtype=np.int16)
    for j, s in enumerate(r.antecedents):
        A[j, [col[i] for i in s]] = 1
    X = Bte[others].to_numpy(np.int16)
    match = (X @ A.T) == A.sum(1)
    conf = np.where(match, r.confidence.to_numpy(), -1)
    best_rule = conf.argmax(axis=1)
    covered = conf.max(axis=1) >= 0
    cons = np.array([next(iter(s)) for s in r.consequents])
    mode = Btr[targets].sum().idxmax()
    pred = np.where(covered, cons[best_rule], mode)
    ok = truth.notna().to_numpy()
    t, p = truth.to_numpy()[ok], pred[ok]
    return {"attribute": attr_prefix, "levels": len(targets), "masked": int(ok.sum()), "rules": len(r),
            "coverage": covered[ok].mean(), "accuracy (rules)": (t == p).mean(),
            "accuracy (mode baseline)": (t == mode).mean(), "macro-F1 (rules)": f1_score(t, p, average="macro"),
            "macro-F1 (mode baseline)": f1_score(t, np.full(len(t), mode), average="macro"), "mode": mode}

imp = pd.DataFrame([impute_eval("price"), impute_eval("body")])
imp.to_csv(ps.TAB_DIR / "m4_imputation.csv", index=False)
imp.round(3)

,attribute,levels,masked,rules,coverage,accuracy (rules),accuracy (mode baseline),macro-F1 (rules),macro-F1 (mode baseline),mode
0,price,3,2769,6393,1.0,0.752,0.351,0.724,0.173,price=low
1,body,6,2769,6082,1.0,0.650,0.642,0.155,0.130,body=suv_crossover


**Imputation results.**
- **Price tertile:** rules recover it for 75.2% of the masked sellers against 35.1% for the mode baseline (macro-F1 0.72 vs 0.17). The transactions contain strong price determinants (mileage, horsepower, new/used profile, savings), as the regression in Module 3 already showed.
- **Body type:** rules barely beat the mode (65.0% vs 64.2%; macro-F1 0.16 vs 0.13). The transactions do not contain the vehicle-dimension variables (length, height, legroom) that actually identify body type: only horsepower is encoded. The remaining items (price, dealer type, colour) carry almost no information about it, so the best rules mostly re-predict `suv_crossover`.
- **Coverage** is 100% for both attributes.
- **Conclusion:** rule-based imputation works well when the encoded attributes are informative about the target attribute, and fails, honestly reported, when they are not.

### 4.3 Profiling the Module 2 segments with rules
The K-Means (k = 4) segment of each seller is added as an item (`segment=C0…C3`), and we mine rules `A → segment=Ck` (min_sup 0.02, |A| ≤ 3). The highest-confidence non-redundant rules describe each segment in the vocabulary of items. This links the unsupervised segmentation of Module 2 with interpretable patterns.

In [14]:
Bseg = encode_items(df, enc_all, rating=True, extra={"segment": "segment=C" + clusters.astype(str)})
fi_s = apriori(Bseg, min_support=0.02, use_colnames=True, max_len=4, low_memory=True)
rs = association_rules(fi_s, num_itemsets=len(Bseg), metric="confidence", min_threshold=0.5,
                       return_metrics=["support", "confidence", "lift"])
rs = rs[rs.consequents.map(lambda s: len(s) == 1 and next(iter(s)).startswith("segment="))
        & rs.antecedents.map(lambda s: not any(i.startswith("segment=") for i in s))]
seg_rules = []
for k in range(4):
    x = rs[rs.consequents == frozenset([f"segment=C{k}"])]
    x = prune_redundant(x.sort_values(["confidence", "support"], ascending=False).head(200)).sort_values(["confidence", "support"], ascending=False).head(3)
    seg_rules.append(x.assign(rule=x.apply(fmt_rule, axis=1)))
seg_rules = pd.concat(seg_rules)[["rule", "support", "confidence", "lift"]]
seg_rules.to_csv(ps.TAB_DIR / "m4_segment_rules.csv", index=False)
seg_rules.round(3)

,rule,support,confidence,lift
29510,dealer=franchise ∧ dom=mid ∧ savings=none → segment=C0,0.122,1.000,2.731
38242,dom=mid ∧ inventory=mixed ∧ savings=none → segment=C0,0.121,1.000,2.731
37124,dom=low ∧ inventory=mixed ∧ savings=none → segment=C0,0.100,1.000,2.731
37584,dom=low ∧ price=high ∧ savings=high → segment=C1,0.021,0.979,3.707
29215,dealer=franchise ∧ dom=low ∧ savings=high → segment=C1,0.027,0.977,3.697
31005,dealer=franchise ∧ inventory=used_only ∧ size=mid → segment=C1,0.026,0.975,3.691
36548,dom=high ∧ price=low ∧ span=high → segment=C2,0.046,0.838,9.048
36300,dom=high ∧ mileage=high ∧ span=high → segment=C2,0.044,0.835,9.018
36674,dom=high ∧ size=low ∧ span=high → segment=C2,0.035,0.801,8.653
40227,hp=low ∧ mileage=high ∧ span=low → segment=C3,0.070,0.985,3.555


**Segment rules.** Each K-Means segment from Module 2 is characterized by short, near-deterministic rules that agree with the centroid profiles:
- **C0 (franchise new-car dealers):** `dealer=franchise ∧ savings=none ∧ dom=mid` → C0 with confidence 1.00.
- **C1 (mid-market used dealers):** discounting (`savings=high`) and fast turnover (`dom=low`) at high prices, or franchise dealers with used-only, mid-size inventories (confidence ≈ 0.98).
- **C2 (slow-turnover dealers):** `dom=high ∧ span=high` plus a cheap, old or small inventory (confidence 0.80–0.84). The lift of ≈ 9 reflects how rare and specific this segment is (9% of sellers), and its lower confidence matches its fragility found in Module 2.
- **C3 (budget independents):** `mileage=high ∧ span=low` with low price or low horsepower (confidence ≈ 0.98).

These rules give the Module 2 segments a readable, rule-based definition that can be applied without the distance model.

## 5. Module 4 summary
- **Encoding:** equal-frequency tertiles plus domain bins give a dense, balanced transaction set (50 items, 13.3 per seller). `min_sup` = 0.05 and `min_conf` = 0.6 give a manageable, almost entirely positively-associated rule set (lift > 1 for 99.8% of rules).
- **Patterns:** they rediscover the two main seller types (franchise new-car dealers vs. independent budget used-car dealers) and the strong price determinants. Associations *with the rating* are weak: the best rating rules have support of 1–2% and lift ≈ 2 (for `low` and `high`).
- **Exploitation:**
  - The rating classifier built from class-association rules reaches the Module 3 level (test macro-F1 0.433), once the rules are combined by a fair aggregation.
  - Rule-based imputation is very effective for price (75% vs 35% accuracy) and ineffective for body type, whose determinants are not encoded.
  - Segment rules turn the Module 2 clusters into readable definitions.